## Download and Clean Dataset

Comencemos importando las bibliotecas.

In [ ]:
import keras
import pandas as pd
import numpy as np
from keras.models import Sequential
from keras.layers import Dense
from keras.utils import to_categorical
from sklearn.metrics import mean_squared_error

import warnings
warnings.simplefilter('ignore', FutureWarning)

Descarguemos los datos y leámoslos en un marco de datos <em>pandas</em>.

In [ ]:
concrete_data = pd.read_csv('https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/DL0101EN/labs/data/concrete_data.csv')
concrete_data.head()
concrete_data.shape

Verifiquemos el conjunto de datos para ver si faltan valores.

In [ ]:
concrete_data.describe()
concrete_data.isnull().sum()

## D. Aumentar el número de capas ocultas

Repita la parte B pero utilice en su lugar una red neuronal con lo siguiente

- Tres capas ocultas de 10 nodos cada una y función de activación ReLU.

1. Divida aleatoriamente los datos en un conjunto de entrenamiento y otro de prueba reservando el 30% de los datos para la prueba. Puede utilizar la función 
train_test_splithelper

In [ ]:
from sklearn.model_selection import train_test_split

concrete_data_columns = concrete_data.columns

# Creamos los conjuntos de entrenamiento
predictors = concrete_data[concrete_data_columns[concrete_data_columns != 'Strength']] # todas las columnas excepto Strength
target = concrete_data['Strength'] # Strength columna

# Normalizamos los datos
predictors_norm = (predictors - predictors.mean()) / predictors.std()
predictors_norm.head()

# Guardamos el número de predictores
n_cols = predictors_norm.shape[1]

# Dividimos los datos
X_train, X_test, y_train, y_test = train_test_split(predictors_norm, target, test_size=0.3, random_state=4)
print('Train set', X_train.shape, y_train.shape)
print('Test set', X_test.shape, y_test.shape)

# Definimos la red con las especificaciones
def regression_nodel():
    # Creamos el modelo con las especificaciones
    model = Sequential()
    model.add(Dense(10, activation='relu', input_shape=(n_cols,)))
    model.add(Dense(10, activation='relu'))
    model.add(Dense(10, activation='relu'))
    model.add(Dense(1))

    # Compilamos; optimizamos y usamos el error
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# Construimos el model
model = regression_nodel()

2. Entrene el modelo en los datos de entrenamiento utilizando 100 épocas.

In [ ]:
model.fit(X_train, y_train, epochs=100, verbose=0)

3. Evalúe el modelo en los datos de prueba y calcule el error cuadrático medio entre la resistencia del hormigón predicha y la resistencia real del hormigón. Puede utilizar la función mean_squared_error de Scikit-learn.

In [ ]:
# Evaluamos el modelo
scores = model.evaluate(X_test, y_test, verbose=0)
print('Accuracy: {}% \n Error: {}'.format(scores[1], 1 - scores[1]))        

# Hacemos la predicción para poder calcular el error
predictions = model.predict(X_test)

# Calculamos el MSE
mse = mean_squared_error(y_test, predictions)
print(f"MSE: {mse}")

4. Repita los pasos 1 - 3, 100 veces, es decir, cree una lista de 100 errores medios al cuadrado.

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Lista para almacenar los errores medios al cuadrado (MSE)
mse_list = []

# Número de repeticiones
n_repeats = 100

for i in range(n_repeats):
    # Paso 1: Dividimos los datos de nuevo en cada iteración
    X_train, X_test, y_train, y_test = train_test_split(predictors_norm, target, test_size=0.3, random_state=i)
    
    # Paso 2: Definimos y entrenamos el modelo en cada iteración
    model = Sequential()
    model.add(Dense(10, activation='relu', input_shape=(n_cols,)))
    model.add(Dense(10, activation='relu'))
    model.add(Dense(10, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mean_squared_error')
    
    # Entrenamos el modelo
    model.fit(X_train, y_train, epochs=100, verbose=0)
    
    # Paso 3: Hacemos la predicción y calculamos el MSE
    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    
    # Guardamos el MSE en la lista
    mse_list.append(mse)

5. Informe de la media y la desviación estándar de los errores medios al cuadrado.

In [ ]:
# Calcular la media y la desviación estándar de los MSE
mean_mse = np.mean(mse_list)
std_mse = np.std(mse_list)

# Imprimir los resultados
print(f"Media del error cuadrático medio (MSE): {mean_mse}")
print(f"Desviación estándar del MSE: {std_mse}")

6. ¿Cómo se compara la media de los errores medios al cuadrado con la del paso C?

In [ ]:
# Supongamos que tienes los resultados de ambos pasos

mean_mse_no_norm = 0.05  # Media de MSE sin normalizar
mean_mse_norm = 0.02     # Media de MSE con datos normalizados

# Comparación simple
print(f"MSE sin normalizar: {mean_mse_no_norm}")
print(f"MSE con normalización: {mean_mse_norm}")

if mean_mse_norm < mean_mse_no_norm:
    print("El modelo con datos normalizados tiene un menor error.")
else:
    print("El modelo sin normalizar tiene un menor error.")